# 03 正式 T5 训练与结果导出

本节实现 Mengzi-T5-Base 的正式训练与评估流程：

1. 读取全量数据并过滤异常样本；
2. 配置训练参数；
3. 微调 Mengzi-T5-Base；
4. 保存模型、训练日志和 loss 曲线；
5. 在验证集上生成预测；
6. 计算 BLEU-1 到 BLEU-4；
7. 导出报告所需的结果表格。

## 0. 项目路径与目录设置

In [42]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

/Users/yunye/Documents/课程/大三下/机器学习基础/大作业/t5_qa_project/outputs/small_sample_t5/small_sample_eval_metrics.json
{
  "eval_loss": 2.9081978797912598,
  "eval_bleu_1": 0.11155507306108173,
  "eval_bleu_2": 0.08481909873427211,
  "eval_bleu_3": 0.05387776900240414,
  "eval_bleu_4": 0.03334072470277048,
  "eval_prediction_avg_len": 8.125
}


## 1. 导入依赖

In [43]:
import math
import random
from collections import Counter
from inspect import signature

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

import transformers
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("cuda", torch.cuda.is_available())
print("mps", torch.backends.mps.is_available())

torch 2.11.0
transformers 5.9.0
cuda False
mps True


## 2. 实验配置

`RUN_MODE` 提供两种模式：

- `cpu_safe`：训练样本与 step 数较小的安全配置，用于产出可解释的基础结果；
- `gpu_full`：使用全量训练集的完整训练配置。

In [44]:
DATA_DIR = PROJECT_DIR / "data" / "DuReaderQG"
TRAIN_PATH = DATA_DIR / "train.json"
DEV_PATH = DATA_DIR / "dev.json"
MODEL_NAME_OR_PATH = "Langboat/Mengzi-T5-Base"  # 首次运行自动从 Hugging Face 下载

RUN_MODE = "cpu_safe"  # 可改为 "gpu_full"
SEED = 42
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 32

if RUN_MODE == "cpu_safe":
    TRAIN_SAMPLE_SIZE = 1024
    DEV_SAMPLE_SIZE = 256
    MAX_STEPS = 300
    NUM_TRAIN_EPOCHS = None
    TRAIN_BATCH_SIZE = 1
    EVAL_BATCH_SIZE = 4
    EVAL_STEPS = 50
    SAVE_STEPS = 100
elif RUN_MODE == "gpu_full":
    TRAIN_SAMPLE_SIZE = None
    DEV_SAMPLE_SIZE = None
    MAX_STEPS = -1
    NUM_TRAIN_EPOCHS = 3
    TRAIN_BATCH_SIZE = 4
    EVAL_BATCH_SIZE = 8
    EVAL_STEPS = 500
    SAVE_STEPS = 500
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

LEARNING_RATE = 5e-5
OUTPUT_DIR = PROJECT_DIR / "outputs" / f"t5_{RUN_MODE}"
FIGURE_DIR = PROJECT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)

config_summary = {
    "run_mode": RUN_MODE,
    "train_sample_size": TRAIN_SAMPLE_SIZE,
    "dev_sample_size": DEV_SAMPLE_SIZE,
    "max_steps": MAX_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "learning_rate": LEARNING_RATE,
}
print(json.dumps(config_summary, ensure_ascii=False, indent=2))

{
  "run_mode": "cpu_safe",
  "train_sample_size": 1024,
  "dev_sample_size": 256,
  "max_steps": 300,
  "num_train_epochs": null,
  "train_batch_size": 1,
  "eval_batch_size": 4,
  "max_input_length": 512,
  "max_target_length": 32,
  "learning_rate": 5e-05
}


## 3. 读取、过滤、抽样

正式评估时过滤空答案样本，因为 BLEU 计算需要参考答案。

In [45]:
def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["_line_no"] = line_no
            rows.append(row)
    return rows


def keep_valid_qa(row):
    return bool(row.get("context")) and bool(row.get("question")) and bool(row.get("answer"))


def sample_rows(rows, sample_size):
    rows = list(rows)
    random.Random(SEED).shuffle(rows)
    if sample_size is None:
        return rows
    return rows[:sample_size]


train_all = read_jsonl(TRAIN_PATH)
dev_all = read_jsonl(DEV_PATH)
train_valid = [row for row in train_all if keep_valid_qa(row)]
dev_valid = [row for row in dev_all if keep_valid_qa(row)]

train_rows = sample_rows(train_valid, TRAIN_SAMPLE_SIZE)
dev_rows = sample_rows(dev_valid, DEV_SAMPLE_SIZE)

print("train all/valid/used:", len(train_all), len(train_valid), len(train_rows))
print("dev all/valid/used:", len(dev_all), len(dev_valid), len(dev_rows))

Task was destroyed but it is pending!
task: <Task pending name='Task-380' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/yunye/Library/Python/3.11/lib/python/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-381' coro=<Kernel.shell_main() running at /Users/yunye/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/yunye/Library/Python/3.11/lib/python/site-packages/zmq/eventloop/zmqstream.py:563]>
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/codeop.py:125: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  codeob = compile(source, filename, symbol, flags, True)
Task was destroyed but it is pending!
task: <Task pending name='Task-381' coro=<Kernel.shell_main() running at /Users/yunye/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


train all/valid/used: 14520 14520 1024
dev all/valid/used: 984 983 256


## 4. 加载本地模型和 tokenizer

In [46]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME_OR_PATH)

# 本地 checkpoint 会提示 tie_word_embeddings warning。这里显式记录配置，不改原始模型文件。
print("tie_word_embeddings:", getattr(model.config, "tie_word_embeddings", None))
print("parameters:", sum(p.numel() for p in model.parameters()))

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


tie_word_embeddings: True
parameters: 247577856


## 5. Dataset 与输入格式

In [47]:
def build_model_input(row):
    return f"question: {row['question']} context: {row['context']}"


class QADataset(Dataset):
    def __init__(self, rows, tokenizer, max_input_length=512, max_target_length=32):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        model_inputs = self.tokenizer(
            build_model_input(row),
            max_length=self.max_input_length,
            truncation=True,
        )
        labels = self.tokenizer(
            text_target=row["answer"],
            max_length=self.max_target_length,
            truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


train_dataset = QADataset(train_rows, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH)
dev_dataset = QADataset(dev_rows, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

print(build_model_input(train_rows[0])[:300])
print("target:", train_rows[0]["answer"])

question: 一个银行卡可以绑定几个微信 context: 微信支付规则1、绑定银行卡时，需要验证持卡人本人的实名信息，即｛姓名，身份证号｝的信息。2、一个微信号只能绑定一个实名信息，绑定后实名信息不能更改，解卡不删除实名绑定关系。3、同一身份证件号码只能注册最多10个（包含10个）微信支付；4、一张银行卡（含信用卡）最多可绑定3个微信号；5、一个微信号最多可绑定10张银行卡（含信用卡）；6、一个微信帐号中的支付密码只能设置一个；7、银行卡无需开通网银（中国银行、工商银行除外），只要在银行中有预留手机号码，即可绑定微信支付。注：一旦绑定成功，该微信号无法绑定其他姓名的银行卡/信用卡，请谨
target: 3个


## 6. BLEU 计算函数

In [48]:
def char_tokens(text):
    return [ch for ch in str(text).strip() if not ch.isspace()]


def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def sentence_bleu_char(prediction, reference, max_n=4, smooth=1e-9):
    pred_tokens = char_tokens(prediction)
    ref_tokens = char_tokens(reference)
    if not pred_tokens or not ref_tokens:
        return 0.0
    precisions = []
    for n in range(1, max_n + 1):
        pred_counts = ngram_counts(pred_tokens, n)
        ref_counts = ngram_counts(ref_tokens, n)
        total = sum(pred_counts.values())
        if total == 0:
            precisions.append(smooth)
            continue
        overlap = sum(min(count, ref_counts[gram]) for gram, count in pred_counts.items())
        precisions.append((overlap + smooth) / (total + smooth))
    pred_len = len(pred_tokens)
    ref_len = len(ref_tokens)
    bp = 1.0 if pred_len > ref_len else math.exp(1 - ref_len / pred_len)
    return bp * math.exp(sum(math.log(p) for p in precisions) / max_n)


def bleu_scores(predictions, references):
    scores = {}
    for n in range(1, 5):
        values = [sentence_bleu_char(pred, ref, max_n=n) for pred, ref in zip(predictions, references)]
        scores[f"bleu_{n}"] = sum(values) / len(values) if values else 0.0
    return scores


def sanitize_token_ids(token_ids, allow_logits=False):
    """Convert Trainer outputs into safe token id arrays before decoding."""
    import numpy as np

    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.detach().cpu().numpy()
    token_ids = np.asarray(token_ids)

    # Some Trainer/transformers versions may pass logits instead of generated ids.
    if allow_logits and token_ids.ndim == 3:
        token_ids = token_ids.argmax(axis=-1)

    if token_ids.ndim == 1:
        token_ids = token_ids[None, :]

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    vocab_size = len(tokenizer)
    token_ids = np.nan_to_num(token_ids, nan=pad_id, posinf=pad_id, neginf=pad_id)
    token_ids = token_ids.astype("int64", copy=False)

    invalid = (token_ids < 0) | (token_ids >= vocab_size)
    if invalid.any():
        token_ids = token_ids.copy()
        token_ids[invalid] = pad_id
    return token_ids


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = sanitize_token_ids(predictions, allow_logits=True)
    labels = sanitize_token_ids(labels, allow_logits=False)
    labels[labels == -100] = tokenizer.pad_token_id

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = bleu_scores(decoded_preds, decoded_labels)
    scores["prediction_avg_len"] = sum(len(char_tokens(x)) for x in decoded_preds) / len(decoded_preds)
    return scores


## 7. 配置 Trainer

In [49]:
arg_params = signature(Seq2SeqTrainingArguments.__init__).parameters
strategy_key = "eval_strategy" if "eval_strategy" in arg_params else "evaluation_strategy"

training_args_kwargs = {
    "output_dir": str(OUTPUT_DIR / "checkpoints"),
    strategy_key: "steps",
    "eval_steps": EVAL_STEPS,
    "logging_steps": max(10, EVAL_STEPS // 5),
    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
    "save_total_limit": 2,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "predict_with_generate": True,
    "generation_max_length": MAX_TARGET_LENGTH,
    "report_to": [],
    "seed": SEED,
}

if MAX_STEPS and MAX_STEPS > 0:
    training_args_kwargs["max_steps"] = MAX_STEPS
else:
    training_args_kwargs["num_train_epochs"] = NUM_TRAIN_EPOCHS

training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": dev_dataset,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_params = signature(Seq2SeqTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

training_args


Seq2SeqTrainingArguments(output_dir='/Users/yunye/Documents/课程/大三下/机器学习基础/大作业/t5_qa_project/outputs/t5_cpu_safe/checkpoints', per_device_train_batch_size=1, num_train_epochs=3.0, max_steps=300, learning_rate=5e-05, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_steps=0, optim=<OptimizerNames.ADAMW_TORCH_FUSED: 'adamw_torch_fused'>, optim_args=None, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=1, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=False, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=False, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=None, auto_find_batch_size=False, logging_strategy=<IntervalSt

## 8. 开始训练

In [ ]:
train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR / "final_model"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "final_model"))

train_result

/Users/yunye/Library/Python/3.11/lib/python/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Bleu 1,Bleu 2,Bleu 3,Bleu 4,Prediction Avg Len
50,2.282206,1.814138,0.321335,0.269770,0.208478,0.152482,6.796875
100,1.511906,1.652614,0.462132,0.415615,0.324488,0.256056,5.722656
150,1.346890,1.665190,0.453989,0.424946,0.332257,0.279942,5.902344
200,1.068456,1.503412,0.492528,0.459100,0.383042,0.333056,4.660156
250,0.916392,1.344679,0.542737,0.503059,0.432452,0.358338,5.113281
300,0.839392,1.291695,0.556122,0.519670,0.438773,0.366764,5.445312


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/yunye/Library/Python/3.11/lib/python/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/yunye/Library/Python/3.11/lib/python/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 9. 训练日志与 loss 曲线

In [ ]:
log_history = trainer.state.log_history
log_path = OUTPUT_DIR / "trainer_log_history.json"
log_path.write_text(json.dumps(log_history, ensure_ascii=False, indent=2), encoding="utf-8")

train_loss = [(item["step"], item["loss"]) for item in log_history if "loss" in item]
eval_loss = [(item["step"], item["eval_loss"]) for item in log_history if "eval_loss" in item]

plt.figure(figsize=(7, 4))
if train_loss:
    plt.plot([x for x, _ in train_loss], [y for _, y in train_loss], marker="o", label="train loss")
if eval_loss:
    plt.plot([x for x, _ in eval_loss], [y for _, y in eval_loss], marker="s", label="eval loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.title(f"T5 Fine-tuning Loss ({RUN_MODE})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
loss_figure_path = FIGURE_DIR / f"t5_{RUN_MODE}_loss.png"
plt.savefig(loss_figure_path, dpi=200)
plt.show()

log_path, loss_figure_path

## 10. 验证集评估

In [ ]:
eval_metrics = trainer.evaluate()
metrics_path = OUTPUT_DIR / "eval_metrics.json"
metrics_path.write_text(json.dumps(eval_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
eval_metrics

## 11. 导出预测结果

在验证集上生成预测样例，用于结果的案例分析。

In [ ]:
def generate_answers(rows, batch_size=EVAL_BATCH_SIZE, max_new_tokens=MAX_TARGET_LENGTH):
    model.eval()
    model_device = next(model.parameters()).device
    predictions = []
    for start in range(0, len(rows), batch_size):
        batch_rows = rows[start:start + batch_size]
        inputs = tokenizer(
            [build_model_input(row) for row in batch_rows],
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )
        inputs = {key: value.to(model_device) for key, value in inputs.items()}
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
        predictions.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
    return predictions


prediction_rows = dev_rows[:min(100, len(dev_rows))]
predictions = generate_answers(prediction_rows)
references = [row["answer"] for row in prediction_rows]
prediction_scores = bleu_scores(predictions, references)

prediction_df = pd.DataFrame({
    "id": [row.get("id") for row in prediction_rows],
    "question": [row["question"] for row in prediction_rows],
    "reference": references,
    "prediction": predictions,
    "context": [row["context"] for row in prediction_rows],
})

prediction_csv_path = OUTPUT_DIR / "prediction_samples.csv"
prediction_jsonl_path = OUTPUT_DIR / "prediction_samples.jsonl"
prediction_df.to_csv(prediction_csv_path, index=False, encoding="utf-8-sig")
with prediction_jsonl_path.open("w", encoding="utf-8") as f:
    for item in prediction_df.to_dict(orient="records"):
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(prediction_scores)
prediction_df.head(10)

## 12. BLEU 指标图表

In [ ]:
bleu_keys = ["bleu_1", "bleu_2", "bleu_3", "bleu_4"]
bleu_values = [prediction_scores[key] for key in bleu_keys]

plt.figure(figsize=(6, 4))
plt.bar([key.upper().replace("_", "-") for key in bleu_keys], bleu_values)
plt.ylim(0, max(0.1, max(bleu_values) * 1.2))
plt.ylabel("score")
plt.title(f"Prediction BLEU Scores ({RUN_MODE})")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
bleu_figure_path = FIGURE_DIR / f"t5_{RUN_MODE}_bleu.png"
plt.savefig(bleu_figure_path, dpi=200)
plt.show()

summary = {
    **config_summary,
    "eval_metrics": eval_metrics,
    "prediction_sample_bleu": prediction_scores,
    "prediction_sample_size": len(prediction_rows),
}
summary_path = OUTPUT_DIR / "experiment_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

bleu_figure_path, summary_path